# 02 - Data Cleaning
In this step, we transform raw database strings into usable data structures while also removing any errors or inconsistencies in our dataset.

In [ ]:
import sqlite3
import pandas as pd
import os
import json

In [ ]:
db_path = os.path.join("..", "data", "raw", "jobs.db")
conn = sqlite3.connect(db_path)

df = pd.read_sql_query("SELECT * FROM jobs", conn)
conn.close()

## Column Selection
We will remove columns that are not needed for analysis. This includes database IDs and job descriptions. This makes our dataset lighter and easier to work with.

In [ ]:
irrelevant_cols = ["id", "description", "url"]
df = df.drop(columns=irrelevant_cols)

## Safety Fill for Categorical Columns
Even though most of our data was successfully captured, a few jobs are missing values for the columns seniority_level, employment_type, contract_duration and remote_type.

To ensure our analysis is consistent and our charts don't have "gaps", we will fill any missing values with the label **"Not Specified"**, since, for those columns, there are not a lot of missing values.

In [ ]:
categorial_columns = [
    "seniority_level",
    "employment_type",
    "contract_duration",
    "remote_type"
]

df[categorial_columns] = df[categorial_columns].fillna("Not Specified")

df[categorial_columns].isna().sum()

## Salary Outlier Investigation
Our initial inspection showed mostly ridiculously high salaries (up to R$132k/month). Let's look at the top 10 highest-paying jobs to see if they are realistic (based on company and seniority level) or if the AI is making mistakes.

In [ ]:
df.sort_values('monthly_salary_max_brl', ascending=False)[['title', 'company', 'monthly_salary_max_brl', 'seniority_level']].head()

## Dropping Salary Columns
Our investigation showed that the salary data is both sparse (missing in ~84% of cases) and unreliable (Jr Software Engineer earning R$132k/month). 

To ensure the integrity of our career intelligence, we will remove those columns entirely and focus our analysis on **skills, remote trends and seniority**.

In [ ]:
cols_to_drop = ['monthly_salary_min_brl', 'monthly_salary_max_brl']

df = df.drop(columns=cols_to_drop)

## Duplicate Check
Having duplicated jobs will trick our analysis into thinking there are more opportunities and more demand for an specific skill than there actually are. We must check if there are duplicated values.

Sometimes a job is reposted with a slightly different URL or description. We should check for jobs with the same title and same company.

In [ ]:
logic_duplicates = df.duplicated(subset=['title', 'company']).sum()
print(f"Number of duplicated jobs: {logic_duplicates}")

## Category Normalization
Currently, different title might refer to the same thing (e.g. "Fullstack" and "Full-stack"). We will group those into standard names so our charts are clear an accurante.

In [ ]:
cols_to_normalize = [
    'job_category', 
    'seniority_level', 
    'employment_type', 
    'contract_duration', 
    'remote_type', 
    'language'
]

for col in cols_to_normalize:
    df[col] = df[col].astype(str).str.lower().str.strip()
    df[col] = df[col].str.replace('-', '', regex=False).str.replace('.', '', regex=False)
    
    print(f"--- Unique values in {col.upper()} ---")
    print(df[col].unique())
    print("\n")

In [ ]:
master_groups = {
    # --- Categories ---
    # --- Front-end ---
    'frontend': 'front-end',
    'frontend/web design': 'front-end',
    'frontend & qa lead': 'front-end',
    
    # --- Back-end ---
    'backend': 'back-end',
    
    # --- Fullstack ---
    'fullstack': 'fullstack',
    'software development': 'fullstack',
    
    # --- Low-code ---
    'power platform developer': 'lowcode',
    'salesforce developer': 'lowcode',
    'salesforce development': 'lowcode',
    'salesforce analyst': 'lowcode',
    'servicenow architecture': 'lowcode',
    
    # --- Blockchain ---
    'blockchain': 'blockchain',
    
    # --- Mobile ---
    'mobile': 'mobile',
    
    # --- Support ---
    'support & success': 'support',
    'voice coaching': 'support',
    
    # --- QA ---
    'quality assurance': 'qa',
    
    # --- DevOps ---
    'infrastructure & devops': 'devops',
    'devops': 'devops',
    'infraestrutura': 'devops',
    'software configuration': 'devops',
    'security engineering': 'devops',
    
    # --- Cloud ---
    'cloud security': 'cloud',
    
    # --- Data ---
    'data & ai': 'data',
    'data collection': 'data',
    
    # --- AI ---
    'ai training / software engineering': 'ai',
    
    # --- Game Dev ---
    'game development': 'game dev',
    
    # --- DBA ---
    'database administration': 'dba',
    'database engineering': 'dba',
    
    # --- RPA ---
    'automation': 'rpa',
    
    # --- Management ---
    'management': 'other',

    # --- Seniority ---
    'internship': 'intern',

    # --- Remote ---
    'null': 'not specified',
}

for col in cols_to_normalize:
    df[col] = df[col].map(master_groups).fillna("other")

In [ ]:
for col in cols_to_normalize:
    
    print(f"--- Unique values in {col.upper()} ---")
    print(df[col].unique())
    print("\n")

## Tech Market Filtering
Now that we have categorized our jobs, we will remove any listings that were classified as "other". This ensures our analysis is focused on technical roles and excludes non-tech jobs like HR or Sales.

In [ ]:
df = df[df['job_category'] != 'other']

df['job_category'].value_counts()


## Hard Skills Processing
The skills are currently stored as JSON strings. We need to parse them into Python lists and standardize the text (lowercase) so we can count chich technologies are most in-demand.

In [ ]:
df['hard_skills'] = df['hard_skills'].apply(lambda x: json.loads(x) if x and x != 'null' else [])
df['hard_skills'] = df['hard_skills'].apply(lambda x: [skill.lower().strip() for skill in x])
df['hard_skills'] = df['hard_skills'].apply(lambda x: list(set(x)))
df[['title', 'hard_skills']].head()

## Saving Clean Dataset
The dataset was cleaned. We should save it so we can keep the original data as it is while still having the clean dataset for the next steps.

In [ ]:
df.to_pickle(os.path.join("..", "data", "processed", "clean_jobs.pkl"))